In [1]:
import importlib.util
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("bitsandbytes installed:", importlib.util.find_spec("bitsandbytes") is not None)
print("accelerate installed:", importlib.util.find_spec("accelerate") is not None)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB",
    )
else:
    raise RuntimeError("CUDA is not available!")

i:\Capstone\knowledge-conflict-pilot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.11.0+cu128
Transformers: 5.24.0
CUDA available: True
bitsandbytes installed: True
accelerate installed: True
GPU: NVIDIA GeForce RTX 4060
VRAM: 8.0 GB


In [ ]:
import gc
import json
import os
import time
from pathlib import Path

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "unsloth/gemma-2-9b-it-bnb-4bit"


MODEL_TAG = MODEL_NAME.split("/")[-1]
DATA_DIR = Path("data")
OUTPUT_DIR = Path(r"I:\Capstone\knowledge-conflict-pilot\outputs") / MODEL_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_NEW_TOKENS = 200 

print("Selected model:", MODEL_NAME)
print("Output folder:", OUTPUT_DIR)

Selected model: unsloth/gemma-2-9b-it-bnb-4bit
Output folder: I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

model.eval()
torch.cuda.reset_peak_memory_stats()

print(f"Model loaded successfully: {MODEL_TAG}")

W1123 22:01:10.655000 1112 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W1123 22:01:11.168000 1112 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 464/464 [00:03<00:00, 119.23it/s]


Model loaded successfully: gemma-2-9b-it-bnb-4bit


In [ ]:
def ask_model(question: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    
    messages = [
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    input_length = inputs["input_ids"].shape[1]
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=pad_id,
        )

    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True,
    )

    return response.strip()

In [5]:
sanity_questions = {
    "English": "What is the capital of Bangladesh? Answer in English.",
    "Bangla": "বাংলাদেশের রাজধানীর নাম কী? বাংলায় উত্তর দাও।",
    "Banglish": "Bangladesh er rajdhani ki? Banglish e uttor dao?",
}

for language, question in sanity_questions.items():
    print(f"\n{'=' * 50}")
    print(language)
    print("=" * 50)

    answer = ask_model(question)

    print("Question:", question)
    print("Answer:", answer)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



English


[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the capital of Bangladesh? Answer in English.
Answer: The capital of Bangladesh is **Dhaka**.

Bangla


[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: বাংলাদেশের রাজধানীর নাম কী? বাংলায় উত্তর দাও।
Answer: বাংলাদেশের রাজধানী হল **ঢাকা**।

Banglish
Question: Bangladesh er rajdhani ki? Banglish e uttor dao?
Answer: Bangladesh-er rajdhani **Dhaka**.


In [6]:
print(
    "GPU Memory Allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)
print(
    "GPU Memory Reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB",
)
print(
    "Peak GPU Memory Reserved:",
    round(torch.cuda.max_memory_reserved() / 1024**3, 2),
    "GB",
)

GPU Memory Allocated: 5.72 GB
GPU Memory Reserved: 5.77 GB
Peak GPU Memory Reserved: 5.77 GB


In [7]:
def run_qa_split(input_filename: str, output_filename: str, verbose: bool = True):
    input_path = DATA_DIR / input_filename
    output_path = OUTPUT_DIR / output_filename

    with open(input_path, "r", encoding="utf-8") as f:
        questions = json.load(f)

    results = []
    start_time = time.time()

    for i, item in enumerate(questions, start=1):
        question = item["question"]
        answer = ask_model(question)

        if verbose:
            print(f"\nQuestion {i}/{len(questions)}")
            print("Q:", question)
            print("A:", answer)

        results.append({
            "question": question,
            "answer": answer,
        })

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

    elapsed = time.time() - start_time
    print(f"\nSaved {len(results)} answers to: {output_path} ({elapsed:.1f}s)")
    return results

In [8]:
english_results = run_qa_split(
    input_filename="questions_english.json",
    output_filename="english_base_answer.json",
)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question 1/50
Q: What is the capital of Australia?
A: The capital of Australia is **Canberra**.


[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question 2/50
Q: Which is the largest country in the world by area?
A: The largest country in the world by area is **Russia**.


[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question 3/50
Q: Who was the first person to walk on the Moon?
A: **Neil Armstrong** was the first person to walk on the Moon. 

He took his famous "one small step" on July 20, 1969, during the Apollo 11 mission.


... (88 outputs left out)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question 48/50
Q: Which scientist proposed the three laws of motion?
A: The three laws of motion were proposed by **Sir Isaac Newton**.


[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question 49/50
Q: In which country did the Olympic Games originate?
A: The Olympic Games originated in **ancient Greece**.

Question 50/50
Q: Which part of the human brain controls balance and coordination?
A: The **cerebellum** is the part of the brain primarily responsible for balance and coordination. 

Here's a little more detail:

* **Cerebellum:** This structure, located at the back of the brain, receives information from the sensory systems, the spinal cord, and other parts of the brain. It uses this information to coordinate voluntary movements, maintain balance, and ensure smooth, precise actions.

Let me know if you have any other questions about the brain!

Saved 50 answers to: I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit\english_base_answer.json (171.0s)


In [9]:
bangla_results = run_qa_split(
    input_filename="questions_bangla.json",
    output_filename="bangla_base_answer.json",
)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question 1/50
Q: অস্ট্রেলিয়ার রাজধানীর নাম কী?
A: অস্ট্রেলিয়ার রাজধানী হল **কানবেরা**।


[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question 2/50
Q: আয়তনের দিক থেকে পৃথিবীর সবচেয়ে বড় দেশ কোনটি?
A: আয়তনের দিক থেকে পৃথিবীর সবচেয়ে বড় দেশ **রাশিয়া**।


[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question 3/50
Q: চাঁদে প্রথম পা রাখা ব্যক্তি কে ছিলেন?
A: চাঁদে প্রথম পা রাখা ব্যক্তি ছিলেন **নেইল আর্মস্ট্রং**। 

তিনি 20 জুলাই, 1969-এ  আপলো 11 মিশনের অংশ হিসেবে চাঁদে প্রথম পা রেখেছিলেন।


... (88 outputs left out)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question 48/50
Q: গতির তিনটি সূত্র কে প্রস্তাব করেছিলেন?
A: গতির তিনটি সূত্র প্রস্তাব করেছিলেন **ইসরাইলের পদার্থবিজ্ঞানী আইজ্যাক নিউটন**।


[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question 49/50
Q: অলিম্পিক গেমসের উৎপত্তি কোন দেশে?
A: অলিম্পিক গেমসের উৎপত্তি **গ্রিস** দেশে। 

প্রাচীন গ্রীসে প্রতি ৪ বছর অন্তর এই গেমস অনুষ্ঠিত হতো।

Question 50/50
Q: মানব মস্তিষ্কের কোন অংশ শরীরের ভারসাম্য ও সমন্বয় নিয়ন্ত্রণ করে?
A: মানব মস্তিষ্কের **মেরুদন্ডু** অংশ শরীরের ভারসাম্য ও সমন্বয় নিয়ন্ত্রণ করে। 

মেরুদন্ডু মস্তিষ্কের নিচের অংশ, যা স্পাইনাল কোর্ডের সাথে সংযুক্ত। এটি শরীরের বিভিন্ন অংশের স্নায়ুকে সংকেত প্রেরণ ও গ্রহণ করে, যা স্থিতি, আন্দোলন, ভারসাম্য, এবং সমন্বয়ের জন্য প্রয়োজন। 

মেরুদন্ডুতে অবস্থিত **সেমি-সিরিব্রাম** নামক অংশটি বিশেষ করে ভারসাম্য ও সমন্বয় নিয়ন্ত্রণ

Saved 50 answers to: I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit\bangla_base_answer.json (242.0s)


In [10]:
banglish_results = run_qa_split(
    input_filename="questions_banglish.json",
    output_filename="banglish_base_answer.json",
)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question 1/50
Q: Australia-r rajdhanir nam ki?
A: Australia-r rajdhanir nam **Canberra** .


[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question 2/50
Q: Ayotoner dik theke prithibir shobcheye boro desh konti?
A: Prithibir shobcheye boro desh **Rusiya**.


[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question 3/50
Q: Chande prothom pa rakha bekti ke chilen?
A: আমি একটি বৃহৎ ভাষা মডেল, তাই আমার কোনো ব্যক্তিগত অভিজ্ঞতা নেই। 

"চন্দ্র প্রথম পাহাড় রক্ষা বেঁচেছিলেন?" এই প্রশ্নের উত্তর দিতে আমার কোনো তথ্য নেই। 

আপনি যদি এই বিষয়ে আরও জানতে চান, তাহলে আমি সুপারিশ করব যে আপনি বিশ্বাসযোগ্য উৎসগুলিতে তথ্য খুঁজে বের করুন।


... (88 outputs left out)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question 48/50
Q: Gotir tinti sutro ke prostab korechilen?
A: Maaf kijiye, par main aapke sawaal ka jawab nahi de sakta. Main ek AI assistant hoon, aur mujhe medical advice nahi dene ka permission hai. 

Agar aapko prostata ke bare mein koi problem hai, to please ek doctor se baat karein.


[transformers] Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question 49/50
Q: Olympic Games-er utpotti kon deshe?
A: Olympic Games-er utpotti kon deshe? 

This question translates to "Which country hosted the Olympic Games?" in Bengali. 

To answer your question, I need more information.  

**Please specify which Olympic Games you are asking about:**

* **Summer Olympics or Winter Olympics?**
* **A specific year?** 

For example, you could ask:

* "Which country hosted the Summer Olympics in 2020?"
* "Which country hosted the Winter Olympics in 1980?"


Once you provide me with more details, I can tell you which country hosted the Olympic Games.

Question 50/50
Q: Manob mostishker kon ongsho shorirer varshamyo o shomonnoy niyontron kore?
A: আমি একটি বৃহৎ ভাষা মডেল, যা গুগল কর্তৃক প্রশিক্ষিত। 

আমি মানুষের মতো চিন্তা বা অনুভব করি না। আমি শুধুমাত্র প্রশ্নের উত্তর দিতে এবং মানুষের সাথে কথা বলতে প্রশিক্ষিত। 

আমি তোমার প্রশ্নের উত্তর দিতে সক্ষম হব যদি তুমি আমাকে স্পষ্টভাবে জিজ্ঞাসা করো।

Saved 50 answers to: I:\Capstone\knowledge-conflict-pilot\ou